# Tools and Permissions — Claude Managed Agents

In modules 01 and 02 the agent ran its tools on its own: you sent a message and read the reply. In this module **your client takes part in the turn**. Two kinds of tool call pause the agent until you answer:

| Pause | The agent emits | You answer with |
|---|---|---|
| A **custom tool** — a tool you define and your code runs | `agent.custom_tool_use` | `user.custom_tool_result` |
| A **built-in tool** whose permission policy asks first | `agent.tool_use` with `evaluated_permission: "ask"` | `user.tool_confirmation` (`allow` / `deny`) |

You'll build an inventory agent that looks stock up through a custom tool backed by Python code, and that needs your approval before every `bash` command.

```
configure the toolset  →  define a custom tool  →  write an approval policy
→  event loop that answers each call as it arrives  →  read stop_reason to know when the turn is over
```

## 1. Setup

Same as the previous modules: install deps and load the API key from the project-root `.env`.

In [1]:
%pip install -q -U "anthropic>=1.13" python-dotenv --user


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

client = anthropic.Anthropic()
print("Client ready")

Client ready


## 2. Create an Environment

Nothing new here — a cloud sandbox. Create it once and reuse the ID.

In [3]:
environment = client.beta.environments.create(
    name="tools-permissions-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
    },
)

print(f"Environment ID: {environment.id}")

Environment ID: env_015XWZ9f5rgW2CFfpzYJLrkT


## 3. Configure the Built-in Toolset

`agent_toolset_20260401` turns on every built-in tool. So far we used it as is; now we tune it with a `configs` array. Each entry names one tool and overrides its settings:

| Tool | `name` | | Tool | `name` |
|---|---|---|---|---|
| Bash | `bash` | | Glob | `glob` |
| Read | `read` | | Grep | `grep` |
| Write | `write` | | Web fetch | `web_fetch` |
| Edit | `edit` | | Web search | `web_search` |

Two kinds of override:

- **`enabled: false`** removes a tool from the agent entirely. Our inventory agent has no business on the web, so we turn off `web_search` and `web_fetch`.
- **`permission_policy`** keeps the tool but controls *when* it runs:

| Policy | Behavior |
|---|---|
| `always_allow` | Runs with no confirmation. **Default for the built-in toolset.** |
| `always_ask` | The session pauses until your client sends `allow` or `deny`. |
| `auto` | The server evaluates each call: runs it, denies it, or pauses for your approval. |

We put `bash` under `always_ask`: the agent can read and write files freely, but every shell command waits for us. (A policy for the whole toolset goes in `default_config.permission_policy`; `configs` entries override it per tool.)

In [4]:
toolset = {
    "type": "agent_toolset_20260401",
    "configs": [
        {"name": "web_search", "enabled": False},
        {"name": "web_fetch", "enabled": False},
        {"name": "bash", "permission_policy": {"type": "always_ask"}},
    ],
}

## 4. Define a Custom Tool

A **custom tool** is a tool the agent can call but **your code runs**. You declare its name, a description, and a JSON Schema for its input; when the agent calls it, your client receives the call, does the work in *your* system, and sends the result back. The agent never sees how it's implemented.

Our tool, `lookup_stock`, reads from a Python dict standing in for a warehouse database. The description matters more than anything else for tool quality — say what the tool does, when to use it, and what each parameter means.

> Custom tools are **not** governed by permission policies: your code decides whether to run them. Names can't start with `mcp__` or `ant__`, or match a built-in tool.

In [5]:
lookup_stock_tool = {
    "type": "custom",
    "name": "lookup_stock",
    "description": (
        "Look up how many units of a product are in the warehouse. "
        "Use this for every stock question; never guess stock levels. "
        "Pass the SKU exactly as the user wrote it, for example A-100. "
        "Returns the unit count, or an error if the SKU is unknown."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "sku": {"type": "string", "description": "Product SKU, for example A-100"},
        },
        "required": ["sku"],
    },
}

# The "warehouse database" — in a real app this would be your own API or DB.
STOCK = {"A-100": 42, "B-200": 0, "C-300": 7}


def lookup_stock(sku: str) -> tuple[str, bool]:
    """Run the tool in our system. Returns (result text, is_error)."""
    if sku not in STOCK:
        return json.dumps({"error": f"Unknown SKU: {sku}"}), True
    return json.dumps({"sku": sku, "units": STOCK[sku]}), False


CUSTOM_TOOLS = {"lookup_stock": lookup_stock}
print(lookup_stock("A-100"), lookup_stock("Z-999"))

('{"sku": "A-100", "units": 42}', False) ('{"error": "Unknown SKU: Z-999"}', True)


## 5. Create the Agent

The `tools` array now has two entries: the configured built-in toolset and our custom tool.

In [6]:
agent = client.beta.agents.create(
    name="Inventory Agent",
    model="claude-opus-5-5",
    system=(
        "You are an inventory assistant. Use the lookup_stock tool for stock levels. "
        "Keep your answers short."
    ),
    tools=[toolset, lookup_stock_tool],
)

print(f"Agent ID : {agent.id}  (v{agent.version})")
for tool in agent.tools:
    print(" -", tool.type, getattr(tool, "name", ""))

Agent ID : agent_01HHYoTNSkA1fYxvwdD9qZ6n  (v1)


 - agent_toolset_20260401 
 - custom lookup_stock


## 6. An Approval Policy

When a `bash` call pauses, *someone* has to decide. In a real application that's often a person clicking Allow or Deny in your UI. To keep this notebook runnable top to bottom, the decision is a small rule: shell commands are fine, but nothing that deletes files.

A denial can carry a `deny_message`. The agent receives it as the tool's result, so it can explain or change course.

> **Try it interactively:** replace the body with `answer = input(f"Allow {command}? [y/N] ")` and return `("allow", None)` when the answer is `y`.

In [7]:
def approve(command: str) -> tuple[str, str | None]:
    """Decide on a bash call. Returns ("allow" | "deny", deny_message)."""
    if "rm " in command:
        return "deny", "Deleting files is not allowed in this workspace."
    return "allow", None

## 7. The Event Loop: Answer Each Call as It Arrives

Module 02's loop only printed text and broke on idle. Now it has to **answer** calls too, and that changes how it decides the turn is over.

When a call needs you, the session goes idle with a `stop_reason`. The full set:

| `stop_reason.type` | Meaning | What the loop does |
|---|---|---|
| `end_turn` | The agent finished its turn | Stop reading |
| `requires_action` | Tool calls are waiting on you; their IDs are in `stop_reason.event_ids` | **Keep reading** — we answer each call as its event arrives |
| `budget_reached` | The session hit its spending cap (module 05) | Stop reading |
| `retries_exhausted`, `refusal` | The session can't continue this turn | Stop reading |

The loop below answers **each call as soon as its event arrives**, not when the idle comes:

- `agent.custom_tool_use` → run the tool, send `user.custom_tool_result` with the event's `id`
- `agent.tool_use` with `evaluated_permission == "ask"` → decide, send `user.tool_confirmation` with the event's `id`

Why not wait for the `requires_action` idle and then answer its `event_ids`? That works for a single agent, and the docs show it. But once a session runs several agents (modules 06–07), it can stay `running` while one of them waits on you, so the idle never comes. Answering on arrival works in both cases — a habit worth building now.

In [8]:
def run_turn(session_id: str, text: str) -> None:
    """Send one user message, answer every tool call, and stream until the turn ends."""
    with client.beta.sessions.events.stream(session_id) as stream:   # stream FIRST
        client.beta.sessions.events.send(
            session_id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": text}]}],
        )

        for event in stream:
            match event.type:
                case "agent.message":
                    for block in event.content:
                        if block.type == "text":
                            print(f"Agent: {block.text}")

                case "agent.custom_tool_use":
                    # A custom tool: run it in our system and send the result back.
                    result, is_error = CUSTOM_TOOLS[event.name](**event.input)
                    print(f"[custom tool] {event.name}({event.input}) -> {result}")
                    client.beta.sessions.events.send(
                        session_id,
                        events=[{
                            "type": "user.custom_tool_result",
                            "custom_tool_use_id": event.id,
                            "content": [{"type": "text", "text": result}],
                            "is_error": is_error,
                        }],
                    )

                case "agent.tool_use":
                    # A built-in tool. Under always_ask it waits for our confirmation.
                    detail = event.input.get("command") or event.input.get("file_path") or ""
                    policy = event.evaluation.type if event.evaluation else "-"
                    print(f"[{event.name}] {detail}  (permission: {event.evaluated_permission}, policy: {policy})")
                    if event.evaluated_permission == "ask":
                        decision, deny_message = approve(detail)
                        confirmation = {
                            "type": "user.tool_confirmation",
                            "tool_use_id": event.id,
                            "result": decision,
                        }
                        if deny_message:
                            confirmation["deny_message"] = deny_message
                        client.beta.sessions.events.send(session_id, events=[confirmation])
                        print(f"    -> {decision}" + (f": {deny_message}" if deny_message else ""))

                case "session.status_idle":
                    reason = event.stop_reason.type
                    if reason == "requires_action":
                        # Calls are waiting on us. We answer each one as its event
                        # arrives, so keep reading until the agent ends its turn.
                        print(f"[idle: requires_action, waiting on {len(event.stop_reason.event_ids)} call(s)]")
                        continue
                    print(f"[idle: {reason}]")
                    break

                case "session.status_terminated":
                    print("[terminated]")
                    break


print("run_turn() ready")

run_turn() ready


## 8. Run It

The request exercises everything: three stock lookups (one with an SKU that doesn't exist), a `bash` command we'll allow, and a delete our policy will deny.

In [9]:
session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    title="Tools and permissions",
)
print(f"Session ID: {session.id}")

Session ID: sesn_01QHfxNt7NjnipzCVPMJhPJ7


In [10]:
run_turn(
    session.id,
    "Check stock for A-100, B-200 and Z-999. Then use bash to write the results to "
    "/workspace/stock.csv (columns sku,units) and show it with cat. "
    "Finally, use bash to delete /workspace/stock.csv.",
)

[custom tool] lookup_stock({'sku': 'A-100'}) -> {"sku": "A-100", "units": 42}


[custom tool] lookup_stock({'sku': 'B-200'}) -> {"sku": "B-200", "units": 0}


[custom tool] lookup_stock({'sku': 'Z-999'}) -> {"error": "Unknown SKU: Z-999"}


[idle: requires_action, waiting on 3 call(s)]
[idle: requires_action, waiting on 2 call(s)]
[idle: requires_action, waiting on 1 call(s)]


[bash] mkdir -p /workspace && printf 'sku,units\nA-100,42\nB-200,0\nZ-999,UNKNOWN\n' > /workspace/stock.csv && cat /workspace/stock.csv  (permission: ask, policy: always_ask)


    -> allow
[idle: requires_action, waiting on 1 call(s)]


[bash] rm /workspace/stock.csv && ls /workspace/stock.csv 2>&1  (permission: ask, policy: always_ask)


    -> deny: Deleting files is not allowed in this workspace.
[idle: requires_action, waiting on 1 call(s)]


Agent: I wrote and displayed the file, but couldn't delete it. The workspace blocks deleting files, so `/workspace/stock.csv` is still there.

- **A-100:** 42 units
- **B-200:** 0 units (out of stock)
- **Z-999:** unknown SKU. The stock system doesn't recognise it, so I put `UNKNOWN` in the units column.

```
sku,units
A-100,42
B-200,0
Z-999,UNKNOWN
```

To remove the file, you'll need to delete it yourself or allow file deletion here.
[idle: end_turn]


### Reading the trace

- **Custom tool calls** — the agent called `lookup_stock` once per SKU, often in parallel. Each call is answered right away. `Z-999` comes back with `is_error: true`, and the agent reports it instead of inventing a number.
- **`requires_action` idles** — you'll often see one *after* we already answered, with a shrinking count of waiting calls. The idle and our answers cross in flight, which is why the loop keeps reading instead of stopping there.
- **Allowed `bash`** — the CSV is written and printed once we send `allow`.
- **Denied `bash`** — the delete never runs. The agent gets `Permission to use bash has been rejected. Rejection message: …` as the tool result, and says so in its reply.
- **`end_turn`** — only now is the turn really over.

## 9. Let the Server Decide: the `auto` Policy

Asking for every call doesn't scale. With `auto`, the server evaluates each call (the tool, its input, and the conversation so far) and:

- **runs it** when it judges the call safe,
- **denies it** when it judges the call high-risk — the agent gets an error result, and your client **can't override it**,
- **pauses for you** (`evaluated_permission: "ask"`) when it can't decide.

Each `agent.tool_use` event records the outcome in `evaluated_permission`, and how it was reached in `evaluation`. Let's move `bash` to `auto` by **updating the agent**. Updates create a new version, and a session pins the version it was created with, so the session above is unaffected.

In [11]:
updated = client.beta.agents.update(
    agent.id,
    version=agent.version,   # optimistic concurrency: fails with 409 if someone else changed it
    tools=[
        {
            "type": "agent_toolset_20260401",
            "configs": [
                {"name": "web_search", "enabled": False},
                {"name": "web_fetch", "enabled": False},
                {"name": "bash", "permission_policy": {"type": "auto"}},
            ],
        },
        lookup_stock_tool,
    ],
)
print(f"Agent v{agent.version} -> v{updated.version}")

auto_session = client.beta.sessions.create(
    agent={"type": "agent", "id": updated.id, "version": updated.version},
    environment_id=environment.id,
    title="Tools and permissions (auto)",
)

run_turn(
    auto_session.id,
    "Use bash to show the current date and the free disk space of /workspace.",
)

Agent v1 -> v2


[bash] date; df -h /workspace  (permission: allow, policy: auto)


Agent: The current date is **Sun Oct 11 14:15:58 UTC 2026**.

`/workspace` sits on the root filesystem (`/dev/vda`), which has **30G free**. The disk is 252G, with 11G used (27%).
[idle: end_turn]


Read the trace: `policy: auto` says the server evaluated the call, and `permission:` gives its verdict. A harmless command like this one typically comes back `allow` and runs with no confirmation from us. If the server can't decide it reports `ask`, and `run_turn` answers with our `approve()` rule like before.

> **`auto` is not a human checkpoint.** A call the server judges safe runs before anyone sees it. Keep `always_ask` on any tool whose calls a person must review.

## Summary

Your client now takes part in the agent's turn:

```
toolset configs        →  disable tools, set permission_policy per tool
custom tool            →  you declare it; your code runs it
agent.custom_tool_use  →  run it, send user.custom_tool_result   (on arrival)
agent.tool_use "ask"   →  decide, send user.tool_confirmation    (on arrival)
session.status_idle    →  requires_action: keep reading
                          end_turn / budget_reached / ...: the turn is over
```

Key takeaways:
- Custom tools run **in your system**; the agent only sees the result you send back
- Permission policies control **when** built-in tools run: `always_allow` (default), `always_ask`, `auto`
- A `deny_message` reaches the agent as the tool result, so it can adapt
- An idle is not always the end: check `stop_reason`, and keep reading on `requires_action`
- Answer each call **when its event arrives** — the pattern that still works once sessions run many agents

### Next steps
- **04-files** — mount files into the sandbox and download what the agent produces
- **05-session-control** — steer, interrupt, reconnect, and cap spend with a session budget
- **06-subagents** — delegate to other agents in session threads
- **07-dynamic-workflows** — let the agent write a workflow that runs many agents in parallel